# 10 — Teste temporal final em 2025

Treina o XGBoost definitivo com 2020–2024 e avalia uma única vez em 2025, mantendo o threshold 0,28 definido anteriormente na validação de 2024.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
from xgboost import XGBClassifier

PASTA_PROJETO = "TCC_PRF"
PASTA_PROCESSED = os.path.join(PASTA_PROJETO, "data", "processed")
PASTA_FINAL = os.path.join(PASTA_PROJETO, "outputs", "modelo_final")
os.makedirs(PASTA_FINAL, exist_ok=True)

df = pd.read_parquet(os.path.join(PASTA_PROCESSED, "prf_acidentes_silver_2020_2025.parquet"))
df["br_categoria"] = df["br"].astype("Int64").astype("string").fillna("Não informado")

In [ ]:
colunas_tracado = [c for c in df.columns if c.startswith("tracado_") and c != "tracado_via"]
categoricas = ["uf","br_categoria","dia_semana","fase_dia","sentido_via","condicao_metereologica","tipo_pista","uso_solo","tipo_acidente"]
numericas = ["km","mes","hora","fim_semana","qtd_caracteristicas_tracado","veiculos"] + colunas_tracado
features = categoricas + numericas
TARGET = "acidente_grave"

proibidas = {"mortos","feridos_graves","feridos_leves","feridos","ilesos","ignorados","pessoas","classificacao_acidente","acidente_fatal"}
assert not any(c in proibidas for c in features)

treino_final = df[df["ano"] <= 2024].copy()
teste_2025 = df[df["ano"] == 2025].copy()

X_train, y_train = treino_final[features], treino_final[TARGET]
X_test, y_test = teste_2025[features], teste_2025[TARGET]

neg = (y_train==0).sum()
pos = (y_train==1).sum()
scale_pos_weight = neg/pos

print("Treino 2020-2024:", len(treino_final))
print("Teste 2025:", len(teste_2025))
print("scale_pos_weight:", scale_pos_weight)

In [ ]:
try:
    encoder = OneHotEncoder(handle_unknown="ignore", min_frequency=100, sparse_output=True)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", min_frequency=100, sparse=True)

preprocessador = ColumnTransformer([
    ("categoricas", encoder, categoricas),
    ("numericas", StandardScaler(), numericas)
])

modelo_final = Pipeline([
    ("preprocessamento", preprocessador),
    ("modelo", XGBClassifier(
        n_estimators=500,
        learning_rate=.05,
        max_depth=6,
        min_child_weight=5,
        subsample=.8,
        colsample_bytree=.8,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42
    ))
])

modelo_final.fit(X_train, y_train)
probabilidades = modelo_final.predict_proba(X_test)[:,1]

THRESHOLD_FINAL = 0.28
predicoes = (probabilidades >= THRESHOLD_FINAL).astype(int)

In [ ]:
accuracy = accuracy_score(y_test, predicoes)
precision = precision_score(y_test, predicoes, zero_division=0)
recall = recall_score(y_test, predicoes, zero_division=0)
f1 = f1_score(y_test, predicoes, zero_division=0)
f2 = fbeta_score(y_test, predicoes, beta=2, zero_division=0)
roc_auc = roc_auc_score(y_test, probabilidades)
pr_auc = average_precision_score(y_test, probabilidades)

matriz = confusion_matrix(y_test, predicoes)
tn, fp, fn, tp = matriz.ravel()

resultado_final = pd.DataFrame({
    "metrica": [
        "accuracy","precision","recall","f1","f2","roc_auc","pr_auc","threshold",
        "verdadeiro_negativo","falso_positivo","falso_negativo","verdadeiro_positivo"
    ],
    "valor": [accuracy,precision,recall,f1,f2,roc_auc,pr_auc,THRESHOLD_FINAL,tn,fp,fn,tp]
})
display(resultado_final)
print(classification_report(y_test, predicoes, digits=4))

matriz_df = pd.DataFrame(
    matriz,
    index=["Real não grave","Real grave"],
    columns=["Previsto não grave","Previsto grave"]
)
display(matriz_df)

In [ ]:
resultado_final.to_csv(
    os.path.join(PASTA_FINAL, "resultado_final_2025.csv"),
    index=False, encoding="utf-8-sig"
)
matriz_df.to_csv(
    os.path.join(PASTA_FINAL, "matriz_confusao_2025.csv"),
    encoding="utf-8-sig"
)

joblib.dump(modelo_final, os.path.join(PASTA_FINAL, "xgboost_modelo_final.joblib"))

configuracao = pd.DataFrame({
    "item": [
        "algoritmo","treino","teste","threshold","criterio_threshold",
        "n_estimators","learning_rate","max_depth","min_child_weight","subsample","colsample_bytree"
    ],
    "valor": [
        "XGBoost","2020-2024","2025",0.28,"Maior F2 na validação de 2024",
        500,0.05,6,5,0.8,0.8
    ]
})
configuracao.to_csv(os.path.join(PASTA_FINAL, "configuracao_final.csv"),
                    index=False, encoding="utf-8-sig")

print("Avaliação final concluída. Nenhuma decisão deve ser reajustada com base em 2025.")